In [1]:
import pandas as pd
import requests
import time

In [2]:
# Cell 2 — load your cleaned dataset
combined = pd.read_csv('../data/rentora_model_ready.csv')  # adjust filename to whatever you actually saved

C:\Users\KUNDAN KUMAR\AppData\Local\Temp\ipykernel_30396\3108402289.py:2: DtypeWarning: Columns (0: size_sqft) have mixed types. Specify dtype option on import or set low_memory=False.
  combined = pd.read_csv('../data/rentora_model_ready.csv')  # adjust filename to whatever you actually saved


In [ ]:
# Cell 3 — get unique locations to query (don't repeat lookups for duplicate coordinates)
unique_locs = combined[['city', 'locality', 'latitude', 'longitude']].drop_duplicates(subset=['latitude', 'longitude'])

print(len(unique_locs))

7765


In [4]:
#Overpass API query function — checks for highway/mall/river/mountain within a radius
import requests
import time

def get_geo_features(lat, lon, radius=2000):
    query = f"""
    [out:json][timeout:15];
    (
      way["highway"~"trunk|primary|motorway"](around:{radius},{lat},{lon});
      node["shop"="mall"](around:{radius},{lat},{lon});
      way["natural"="water"](around:{radius},{lat},{lon});
      node["natural"="peak"](around:{radius},{lat},{lon});
    );
    out count;
    """
    try:
        response = requests.post("https://overpass-api.de/api/interpreter", data=query, timeout=20)
        data = response.json()
        counts = data.get('elements', [{}])[-1].get('tags', {})
        return pd.Series([
            int(counts.get('nodes', 0)) + int(counts.get('ways', 0)) > 0  # crude combined flag for test
        ])
    except Exception as e:
        print(f"Failed at {lat},{lon}: {e}")
        return pd.Series([None])